In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import pandas as pd
import seaborn as sns
import random
import multiprocessing
import glob
import metpy
from metpy.units import units
from matplotlib import gridspec
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # stormeunice is not pip-installed in this kernel
import stormeunice as eun

from matplotlib.ticker import MultipleLocator, FormatStrFormatter

style = eun.style
style.set_style()
%matplotlib inline

random.seed(10)

# Import data

In [2]:
P1 = multiprocessing.Pool(40)

ifs_tracks = P1.map(eun.lagrangian.Lagrange.import_medr_tracks_TE, 
                    glob.glob('/home/l/leach/Attribution/NA-Storms/Nick/Scripts/tracking/TE/TEStitch_[!2]*'))

P1.close()

ifs_tracks = pd.concat(ifs_tracks)
ifs_tracks['date'] = pd.to_datetime(ifs_tracks.loc[:,['year','month','day','hour']])

## add era track in
era_track = pd.read_csv('/home/l/leach/Attribution/NA-Storms/Nick/Scripts/tracking/TE/TEStitch_2022_0',
                        skipinitialspace=True)

era_track['expid'] = 'era5'
era_track['experiment'] = 'era5'
era_track['inidate'] = pd.to_datetime('2022-01-01')
era_track['number'] = 0
era_track['date'] = pd.to_datetime(era_track.loc[:,['year','month','day','hour']])

all_tracks = pd.concat([ifs_tracks,era_track])

eunice_track = era_track.query('track_id==5')

In [3]:
ifs_tracks_filter = ifs_tracks.groupby(['track_id','inidate','experiment','number']).agg(dict(date=min, day=np.size, lon=lambda x: x.iloc[0], lat=lambda x: x.iloc[0], msl=min))
eunice_filter = eunice_track.groupby(['track_id','inidate','experiment','number']).agg(dict(date=min, day=np.size, lon=lambda x: x.iloc[0], lat=lambda x: x.iloc[0], msl=min)).iloc[0]

# starttime criterion - must be detected within 12 hours (before or after) the ERA5 track detectino time
starttime_crit = (ifs_tracks_filter.date - eunice_filter.date).dt.total_seconds().abs()<(12*60*60)
# starting location criterion - must start within 10 degrees of the ERA5 track starting point
startloc_crit = np.sqrt((ifs_tracks_filter.lon-eunice_filter.lon)**2+(ifs_tracks_filter.lat-eunice_filter.lat)**2)<10
# depth criterion - must reach a depth of at least 980 hPa
minmsl_crit = ifs_tracks_filter.msl < 98000

ifs_tracks_filtered = ifs_tracks_filter.loc[starttime_crit&startloc_crit&minmsl_crit]
ifs_eunice_list = ifs_tracks.set_index(['track_id','inidate','experiment','number']).sort_index().loc[ifs_tracks_filtered.index].reset_index()

In [4]:
multi_track_members = ifs_eunice_list.groupby(['inidate','experiment','number','track_id']).agg(dict(year = lambda x: 1)).groupby(['inidate','experiment','number']).count().query('year > 1')

for idx in multi_track_members.index:
    track_dists = ifs_eunice_list.query('inidate=="{}" & experiment=="{}" & number=={}'.format(*idx)).groupby('track_id').apply(lambda x: eun.lagrangian.Lagrange.eunice_dist(x,eunice_track=eunice_track))
    num_reject = track_dists.idxmax()
    ifs_eunice_list_idx = ifs_eunice_list.query('inidate=="{}" & experiment=="{}" & number=={} & track_id=={}'.format(*idx,num_reject)).index
    ifs_eunice_list = ifs_eunice_list.drop(ifs_eunice_list_idx)

In [5]:
ERA5_tracks = pd.read_hdf('/gf3/predict2/AWH012_LEACH_NASTORM/DATA/postproc/tracks/ERA5/EU025/sfc/2022.h5')
ERA5_eunice_track = ERA5_tracks.query('uid=="202202079"')

ERA5_data = xr.open_dataset('/gf3/predict2/AWH012_LEACH_NASTORM/DATA/ERA5/EU025/sfc/2022.nc').sel(time=ERA5_eunice_track.date.values)
ERA5_data['centroid_lon'] = ('time',(ERA5_eunice_track.long*4).round()/4)
ERA5_data['centroid_lat'] = ('time',(ERA5_eunice_track.lat*4).round()/4)

ERA5_eunice_stormframe = ERA5_data.groupby('time').apply(eun.lagrangian.Lagrange.lagrangian_frame)

In [6]:
%%capture 
# Ignore outputs

curr_storms = xr.open_mfdataset('/gf3/predict2/AWH012_LEACH_NASTORM/DATA/MED-R/ENS/EU025/pl/pf/extravars/*.nc',
                                preprocess=lambda x: eun.lagrangian.Lagrange.preproc_to_stormframe_max_deepening(x, ifs_eunice_list=ifs_eunice_list, sfc=False),
                                combine='nested',
                                concat_dim=['inidate'],
                                parallel=True, 
                                engine='netcdf4')
pi_storms = xr.open_mfdataset('/gf3/predict2/AWH012_LEACH_NASTORM/DATA/MED-R/EXP/pi/EU025/pl/pf/extravars/*.nc',
                              preprocess=lambda x: eun.lagrangian.Lagrange.preproc_to_stormframe_max_deepening(x, ifs_eunice_list=ifs_eunice_list, sfc=False),
                              combine='nested',
                              concat_dim=['inidate'],
                              parallel=True, 
                              engine='netcdf4')
incr_storms = xr.open_mfdataset('/gf3/predict2/AWH012_LEACH_NASTORM/DATA/MED-R/EXP/incr/EU025/pl/pf/extravars/*.nc',
                                preprocess=lambda x: eun.lagrangian.Lagrange.preproc_to_stormframe_max_deepening(x, ifs_eunice_list=ifs_eunice_list,sfc=False),
                                combine='nested',
                                concat_dim=['inidate'],
                                parallel=True, 
                                engine='netcdf4')

# Meteorological calculations

In [7]:
# calculate eq potential temperature
experiments = ['pi', 'curr', 'incr']
eq_pot_temp = {}
storms = {'pi' : pi_storms, 'curr' : curr_storms, 'incr' : incr_storms}
pot_temp = {}
for exp in experiments:
    pot_temp[exp] = metpy.calc.potential_temperature(storms[exp].level * units.hPa, storms[exp].t * units.kelvin)
    # r = metpy.calc.relative_humidity_from_specific_humidity(storms[exp].level * units.hPa, storms[exp].t * units.kelvin, storms[exp].tp)
    # dewpoint = metpy.calc.dewpoint_from_relative_humidity(storms[exp].t * units.kelvin, r)

    # eq_pot_temp[exp] = metpy.calc.equivalent_potential_temperature(storms[exp].level * units.hPa, storms[exp].t * units.kelvin, dewpoint)

In [8]:
plev = 850
notmissing = np.load('11_members_with_complete_data_in_z_2022-02-14_plev'+str(plev)+'.npz')

# Significance testing

In [9]:
def generate_boot_2sided(test, comp, n_boot=10000, sign_lev=0.9, diff_larger_0=True):
    
    '''
    Two sample bootstrap for the mean of two datasets
    '''

    x = test.values  # three dimensional array: number, lat, lon
    y = comp.values

    #Brute force
    x_sign =  np.full((x.shape[1], x.shape[2]), 0)
    for lon in range(x.shape[1]):
        for lat in range(x.shape[2]):
            x_boot = np.random.choice(x[:, lon,lat], size=(n_boot, x.shape[0]), replace=True)
            y_boot = np.random.choice(y[:, lon,lat], size=(n_boot, x.shape[0]), replace=True)

            diff = np.mean(y_boot, axis=1) - np.mean(x_boot, axis=1)
            ci_lower = np.percentile(diff, 1-sign_lev)
            ci_upper = np.percentile(diff, sign_lev)
            if diff_larger_0 & (ci_lower>0):
                x_sign[lon, lat] = 1
            elif (not diff_larger_0) & (ci_upper<0):
                x_sign[lon, lat] = 1
    return x_sign

def in_interval(test, bounds):
    return (test > bounds[0]) and (test < bounds[1])

In [10]:
def generate_boot_1sided(test, n_boot=10000, sign_lev=0.9):
    
    '''
    One sample bootstrap for the mean of a dataset
    Q: is the mean different from 0?
    '''

    x = test.values  # three dimensional array: number, lat, lon

    #Brute force
    x_sign =  np.full((x.shape[1], x.shape[2]), 0)
    for lon in range(x.shape[1]):
        for lat in range(x.shape[2]):
            x_boot = np.random.choice(x[:,lon,lat], size=n_boot, replace=True)
            ci_lower = np.nanpercentile(x_boot, 100*(1-sign_lev)/2)
            ci_upper = np.nanpercentile(x_boot, 100*(sign_lev+(1-sign_lev)/2))
            if not in_interval(0, [ci_lower, ci_upper]):
                x_sign[lon, lat] = 1
    return x_sign

In [11]:
# Significance tests for specific humidity
ini = '2022-02-14'
ts = 0
pi = pi_storms.sel(inidate=ini, timestep=ts, level=plev, number=notmissing['notmissing_pi']).squeeze().q
incr = incr_storms.sel(inidate=ini, timestep=ts, level=plev, number=notmissing['notmissing_curr']).squeeze().q
test = incr -pi
sign_test_q = generate_boot_1sided(test)

In [12]:
# Significance tests for vorticity
ini = '2022-02-14'
ts = 0
pi = pi_storms.sel(storm_lat = 0, inidate = ini, timestep = ts).squeeze().squeeze().vo
incr = incr_storms.sel(storm_lat = 0, inidate = ini, timestep = ts).squeeze().squeeze().vo
test = incr - pi
sign_test_vo = generate_boot_1sided(test)

# Save or load data

In [13]:
# save the storm composites for all experiments
# pi_storms.sel(inidate='2022-02-14', timestep=0).to_netcdf('/home/e/ermis/Storm_Eunice/plot_data/storm_composites_pi.nc')
# curr_storms.sel(inidate='2022-02-14', timestep=0).to_netcdf('/home/e/ermis/Storm_Eunice/plot_data/storm_composites_curr.nc')
# incr_storms.sel(inidate='2022-02-14', timestep=0).to_netcdf('/home/e/ermis/Storm_Eunice/plot_data/storm_composites_incr.nc')

# # save significance tests
# np.savetxt('/home/e/ermis/Storm_Eunice/plot_data/sign_test_q.csv', sign_test_q, delimiter=',')
# np.savetxt('/home/e/ermis/Storm_Eunice/plot_data/sign_test_vo.csv', sign_test_vo, delimiter=',')

In [14]:
# # Load the storm composites
# pi_storms = xr.open_dataset('/home/e/ermis/Storm_Eunice/plot_data/storm_composites_pi.nc')
# curr_storms = xr.open_dataset('/home/e/ermis/Storm_Eunice/plot_data/storm_composites_curr.nc')
# incr_storms = xr.open_dataset('/home/e/ermis/Storm_Eunice/plot_data/storm_composites_incr.nc')

# # Load significance tests
# sign_test_q = np.loadtxt('/home/e/ermis/Storm_Eunice/plot_data/sign_test_q.csv', delimiter=',')
# sign_test_vo = np.loadtxt('/home/e/ermis/Storm_Eunice/plot_data/sign_test_vo.csv', delimiter=',')

# Plots

In [ ]:
# setup -------------------------
fig, axes = plt.subplots(1, 4, figsize=(20, 3.3), sharey=True)
vmin = 0
vmax = 5
lev_number = 10
ts = 0
ini = '2022-02-14'
z_min = 10000
z_max = 15000
num_levels_z = 10
w_min = -3
w_max = -0.5
num_levels_w = 2
plev = 850

# type sizes are quoted at style.REF_WIDTH and scaled to this canvas, so they
# print at the same size as the other paper figures once scaled to page width
fs_title = style.scaled(style.FS_TITLE, fig)
fs_label = style.scaled(style.FS_LABEL, fig)
fs_tick = style.scaled(style.FS_TICK, fig)

pl_storms = {'pi': pi_storms, 'curr': curr_storms, 'incr': incr_storms}

# one panel per experiment -------------------------
composites = {}
for ax, experiment in zip(axes, ['pi', 'curr', 'incr']):
    comp = (pl_storms[experiment]
            .sel(inidate=ini, timestep=ts, level=plev,
                 number=notmissing['notmissing_' + experiment])
            .squeeze().mean(dim='number'))
    composites[experiment] = comp

    c = (comp.q*1000).plot.contourf(levels=lev_number, cmap=style.CMAP_SEQ, ax=ax,
                                    add_colorbar=False, vmin=vmin, vmax=vmax)

    # geopotential isolines, in white so they clear the dark end of CMAP_SEQ
    cs = ax.contour(comp.storm_lon.values, comp.storm_lat.values, comp.z,
                    np.linspace(z_min, z_max, num_levels_z),
                    colors=style.CONTOUR_LIGHT, linewidths=0.8, zorder=16)
    plt.clabel(cs, fmt='%d', fontsize=fs_tick)

    # isolines for vertical velocities, in the accent style.py keeps clear of
    # both colourmaps
    ax.contour(comp.storm_lon.values, comp.storm_lat.values, comp.w,
               np.linspace(w_min, w_max, num_levels_w),
               colors=style.accent('orchid'), linewidths=1.2, zorder=19)

    ax.set_title(style.experiment_label(experiment), fontsize=fs_title, fontweight='bold')
    ax.set_xlabel('')  # xarray labels the axes from the variable names
    ax.tick_params(axis='both', which='major', labelsize=fs_tick)

# diff -------------------------
diff = composites['incr'] - composites['pi']
ax = axes[3]
c_diff = (diff.q*1000).plot.contourf(levels=lev_number, cmap=style.CMAP_DIV, ax=ax,
                                     add_colorbar=False, vmin=-1, vmax=1)

# isolines, in ink here: the centre of CMAP_DIV is white
cs = ax.contour(diff.storm_lon.values, diff.storm_lat.values, diff.z,
                np.linspace(-600, 600, num_levels_z),
                colors=style.CONTOUR_DARK, linewidths=0.8, zorder=16)
plt.clabel(cs, fmt='%d', fontsize=fs_tick)

# hatching for significance
ax.contourf(diff.storm_lon.values, diff.storm_lat.values, sign_test_q, alpha=0,
            levels=[-0.5, 0.5, 1], hatches=[None, style.HATCH], zorder=21)

ax.set_title('diff(pi, fut)', fontsize=fs_title, fontweight='bold')
ax.set_xlabel('')
ax.tick_params(axis='both', which='major', labelsize=fs_tick)

# axes -------------------------
axes[0].set_ylabel('lat from storm centre', fontsize=fs_label)
for ax in axes[1:]:
    ax.set_ylabel('')

# y axis number formatting
axes[0].yaxis.set_major_locator(MultipleLocator(5))
axes[0].yaxis.set_major_formatter(FormatStrFormatter('%d'))
axes[0].yaxis.set_minor_locator(MultipleLocator(5))

fig.supxlabel('lon from storm centre', x=0.46, y=-0.03, va='top', fontsize=fs_label)

# panel labels a, b, c, d
style.panel_labels(axes)

# cbars -------------------------
plt.subplots_adjust(bottom=0.1, right=0.8, top=0.9)

cax = plt.axes([0.82, 0.1, 0.008, 0.8])  # left, bottom, width, height
cbar = plt.colorbar(c, cax=cax, format='%.1f')
cbar.set_label(label='specific humidity in g/kg', size=fs_label)
cbar.ax.tick_params(labelsize=fs_tick)

cax_diff = plt.axes([0.9, 0.1, 0.008, 0.8])
cbar_diff = plt.colorbar(c_diff, cax=cax_diff, format='%.1f')
cbar_diff.set_label(label='spec. hum. difference in g/kg', size=fs_label)
cbar_diff.ax.tick_params(labelsize=fs_tick)

style.save(fig, 'PAPER4_1_q', directory='figures', formats=('png', 'pdf', 'eps'))
plt.show()

In [ ]:
# setup -------------------------
fig, axes = plt.subplots(1, 4, figsize=(20, 5), sharey=True)
vmin = -7
vmax = 7
levs = 11
ts = 0
ini = '2022-02-14'
t_min = 230
t_max = 360
num_levels_t = 15
levs_array = np.arange(vmin, vmax+0.5, 0.5)

fs_title = style.scaled(style.FS_TITLE, fig)
fs_label = style.scaled(style.FS_LABEL, fig)
fs_tick = style.scaled(style.FS_TICK, fig)

pl_storms = {'pi': pi_storms, 'curr': curr_storms, 'incr': incr_storms}

# one panel per experiment -------------------------
composites = {}
for ax, experiment in zip(axes, ['pi', 'curr', 'incr']):
    comp = (pl_storms[experiment].sel(storm_lat=0, inidate=ini, timestep=ts)
            .squeeze().mean(dim='number'))
    composites[experiment] = comp

    # vorticity is signed about zero, so it takes the signed colourmap rather
    # than the sequential one used for the humidity panels
    c = (comp.vo*10e3).plot.contourf(levels=levs, cmap=style.CMAP_SIGNED, ax=ax,
                                     center=0, vmax=vmax, add_colorbar=False)

    # potential temperature isolines
    cs = ax.contour(comp.storm_lon.values, comp.level.values,
                    pot_temp[experiment].sel(inidate=ini, storm_lat=0, timestep=ts)
                                        .mean(dim='number').squeeze(),
                    np.linspace(t_min, t_max, num_levels_t),
                    colors=style.CONTOUR_DARK, linewidths=0.8, zorder=16)
    plt.clabel(cs, fmt='%d', fontsize=fs_tick)

    ax.set_title(style.experiment_label(experiment), fontsize=fs_title, fontweight='bold')
    ax.set_xlabel('')  # xarray labels the axes from the variable names
    ax.set_ylim(150, 1000)
    ax.set_yscale('log')
    ax.tick_params(axis='both', which='major', labelsize=fs_tick)
    ax.invert_yaxis()

# diff -------------------------
diff = composites['incr'] - composites['pi']
ax = axes[3]
c_diff = (diff.vo*10e3).plot.contourf(ax=ax, levels=levs, cmap=style.CMAP_DIV,
                                      vmin=-2, vmax=2, center=0, add_colorbar=False)

cs = ax.contour(diff.storm_lon.values, diff.level.values,
                pot_temp['incr'].sel(inidate=ini, storm_lat=0, timestep=ts).mean(dim='number').squeeze()
                - pot_temp['pi'].sel(inidate=ini, storm_lat=0, timestep=ts).mean(dim='number').squeeze(),
                np.linspace(-5, 5, num_levels_t),
                colors=style.CONTOUR_DARK, linewidths=0.8, zorder=16)
plt.clabel(cs, fmt='%d', fontsize=fs_tick)

# hatching for significance
ax.contourf(diff.storm_lon.values, diff.level.values, sign_test_vo, alpha=0,
            levels=[-0.5, 0.5, 1], hatches=[None, style.HATCH], zorder=19)

ax.set_title('diff(pi, fut)', fontsize=fs_title, fontweight='bold')
ax.set_xlabel('')
ax.set_ylim(150, 1000)
ax.set_yscale('log')
ax.tick_params(axis='both', which='major', labelsize=fs_tick)
ax.invert_yaxis()

# axes -------------------------
axes[0].set_ylabel('pressure level', fontsize=fs_label)
for ax in axes[1:]:
    ax.set_ylabel('')

# y axis number formatting: the axis is log-spaced, so it is labelled at the
# standard pressure levels rather than on a fixed step
axes[0].set_yticks([200, 300, 500, 700, 850, 1000])
axes[0].yaxis.set_major_formatter(FormatStrFormatter('%d'))
axes[0].minorticks_off()

fig.supxlabel('lon from storm centre', x=0.46, y=-0.03, va='top', fontsize=fs_label)

# panel labels continue the lettering of the humidity figure above
style.panel_labels(axes, letters='efgh')

# cbars -------------------------
plt.subplots_adjust(bottom=0.1, right=0.8, top=0.9)

cax = plt.axes([0.82, 0.1, 0.008, 0.8])  # left, bottom, width, height
cbar = plt.colorbar(c, cax=cax, format='%.1f')
cbar.set_label(label=r'vorticity in 10$^{-3}$/s', size=fs_label)
cbar.ax.tick_params(labelsize=fs_tick)

cax_diff = plt.axes([0.9, 0.1, 0.008, 0.8])
cbar_diff = plt.colorbar(c_diff, cax=cax_diff, format='%.1f')
cbar_diff.set_label(label=r'vorticity difference in 10$^{-3}$/s', size=fs_label)
cbar_diff.ax.tick_params(labelsize=fs_tick)

style.save(fig, 'PAPER4_2_vo', directory='figures', formats=('png', 'pdf', 'eps'))
plt.show()

# Supplement: Precip changes

In [15]:
%%capture 
# Ignore outputs

curr_storms = xr.open_mfdataset('/gf3/predict2/AWH012_LEACH_NASTORM/DATA/MED-R/ENS/EU025/sfc/pf/*.nc',
                                preprocess=lambda x: eun.lagrangian.Lagrange.preproc_to_stormframe(x, ifs_eunice_list=ifs_eunice_list, sfc=True),
                                combine='nested',
                                concat_dim=['inidate'],
                                parallel=True, 
                                engine='netcdf4')
pi_storms = xr.open_mfdataset('/gf3/predict2/AWH012_LEACH_NASTORM/DATA/MED-R/EXP/pi/EU025/sfc/pf/*.nc',
                              preprocess=lambda x: eun.lagrangian.Lagrange.preproc_to_stormframe(x, ifs_eunice_list=ifs_eunice_list, sfc=True),
                              combine='nested',
                              concat_dim=['inidate'],
                              parallel=True, 
                              engine='netcdf4')
incr_storms = xr.open_mfdataset('/gf3/predict2/AWH012_LEACH_NASTORM/DATA/MED-R/EXP/incr/EU025/sfc/pf/*.nc',
                                preprocess=lambda x: eun.lagrangian.Lagrange.preproc_to_stormframe(x, ifs_eunice_list=ifs_eunice_list,sfc=True),
                                combine='nested',
                                concat_dim=['inidate'],
                                parallel=True, 
                                engine='netcdf4')

In [16]:
curr_storms

,Array,Chunk
Bytes,129.75 MiB,25.63 kiB
Shape,"(3, 1, 48, 36, 81, 81)","(1, 1, 1, 1, 81, 81)"
Count,61575 Tasks,5184 Chunks
Type,float32,numpy.ndarray
,Array,Chunk
Bytes,129.75 MiB,25.63 kiB
Shape,"(3, 1, 48, 36, 81, 81)","(1, 1, 1, 1, 81, 81)"
Count,61575 Tasks,5184 Chunks
Type,float32,numpy.ndarray
,Array,Chunk


In [ ]:
# setup -------------------------
fig, axes = plt.subplots(1, 4, figsize=(20, 3.3), sharey=True)
vmin = 0
vmax = 1
lev_number = 10
ts = 0
ini = '2022-02-14'
z_min = 940
z_max = 1020
num_levels_z = 10

fs_title = style.scaled(style.FS_TITLE, fig)
fs_label = style.scaled(style.FS_LABEL, fig)
fs_tick = style.scaled(style.FS_TICK, fig)

sfc_storms = {'pi': pi_storms, 'curr': curr_storms, 'incr': incr_storms}

# one panel per experiment -------------------------
composites = {}
for ax, experiment in zip(axes, ['pi', 'curr', 'incr']):
    comp = (sfc_storms[experiment]
            .sel(inidate=ini, timestep=ts, number=notmissing['notmissing_' + experiment])
            .squeeze().mean(dim='number'))
    composites[experiment] = comp

    c = (comp.mxtpr*1000).plot.contourf(levels=lev_number, cmap=style.CMAP_SEQ, ax=ax,
                                        add_colorbar=False, vmin=vmin, vmax=vmax)

    # isobars, in white so they clear the dark end of CMAP_SEQ
    cs = ax.contour(comp.storm_lon.values, comp.storm_lat.values, comp.msl/100,
                    np.linspace(z_min, z_max, num_levels_z),
                    colors=style.CONTOUR_LIGHT, linewidths=0.8, zorder=16)
    plt.clabel(cs, fmt='%d', fontsize=fs_tick)

    ax.set_title(style.experiment_label(experiment), fontsize=fs_title, fontweight='bold')
    ax.set_xlabel('')  # xarray labels the axes from the variable names
    ax.tick_params(axis='both', which='major', labelsize=fs_tick)

# diff -------------------------
diff = composites['incr'] - composites['pi']
ax = axes[3]
c_diff = (diff.mxtpr*1000).plot.contourf(levels=lev_number, cmap=style.CMAP_DIV, ax=ax,
                                         add_colorbar=False, vmin=-0.5, vmax=0.5)

# isolines, in ink here: the centre of CMAP_DIV is white
cs = ax.contour(diff.storm_lon.values, diff.storm_lat.values, diff.msl/100,
                np.linspace(-10, 10, num_levels_z),
                colors=style.CONTOUR_DARK, linewidths=0.8, zorder=16)
plt.clabel(cs, fmt='%d', fontsize=fs_tick)

ax.set_title('diff(pi, fut)', fontsize=fs_title, fontweight='bold')
ax.set_xlabel('')
ax.tick_params(axis='both', which='major', labelsize=fs_tick)

# axes -------------------------
axes[0].set_ylabel('lat from storm centre', fontsize=fs_label)
for ax in axes[1:]:
    ax.set_ylabel('')

# y axis number formatting
axes[0].yaxis.set_major_locator(MultipleLocator(5))
axes[0].yaxis.set_major_formatter(FormatStrFormatter('%d'))
axes[0].yaxis.set_minor_locator(MultipleLocator(5))

fig.supxlabel('lon from storm centre', x=0.46, y=-0.03, va='top', fontsize=fs_label)

# panel labels a, b, c, d
style.panel_labels(axes)

# cbars -------------------------
plt.subplots_adjust(bottom=0.1, right=0.8, top=0.9)

cax = plt.axes([0.82, 0.1, 0.008, 0.8])  # left, bottom, width, height
cbar = plt.colorbar(c, cax=cax, format='%.1f')
cbar.set_label(label=r'maximum precip. rate in g/m$^2$/s', size=fs_label)
cbar.ax.tick_params(labelsize=fs_tick)

cax_diff = plt.axes([0.9, 0.1, 0.008, 0.8])
cbar_diff = plt.colorbar(c_diff, cax=cax_diff, format='%.1f')
cbar_diff.set_label(label=r'maximum precip. rate diff. in g/m$^2$/s', size=fs_label)
cbar_diff.ax.tick_params(labelsize=fs_tick)

style.save(fig, 'PAPER-SUPP2_mxtpr', directory='figures', formats=('png', 'pdf', 'eps'))
plt.show()

# Most impactful memer: member 4

In [ ]:
# setup -------------------------
fig, axes = plt.subplots(1, 4, figsize=(20, 3.3), sharey=True)
vmin = 0
vmax = 5
lev_number = 10
ts = 0
ini = '2022-02-16'
z_min = 10000
z_max = 15000
num_levels_z = 10
w_min = -3
w_max = -0.5
num_levels_w = 2
plev = 850
member = 4

fs_title = style.scaled(style.FS_TITLE, fig)
fs_label = style.scaled(style.FS_LABEL, fig)
fs_tick = style.scaled(style.FS_TICK, fig)

pl_storms = {'pi': pi_storms, 'curr': curr_storms, 'incr': incr_storms}

# one panel per experiment -------------------------
members = {}
for ax, experiment in zip(axes, ['pi', 'curr', 'incr']):
    memb = pl_storms[experiment].sel(inidate=ini, timestep=ts, level=plev,
                                     number=member).squeeze()
    members[experiment] = memb

    c = (memb.q*1000).plot.contourf(levels=lev_number, cmap=style.CMAP_SEQ, ax=ax,
                                    add_colorbar=False, vmin=vmin, vmax=vmax)

    # geopotential isolines, in white so they clear the dark end of CMAP_SEQ
    cs = ax.contour(memb.storm_lon.values, memb.storm_lat.values, memb.z,
                    np.linspace(z_min, z_max, num_levels_z),
                    colors=style.CONTOUR_LIGHT, linewidths=0.8, zorder=16)
    plt.clabel(cs, fmt='%d', fontsize=fs_tick)

    # isolines for vertical velocities
    ax.contour(memb.storm_lon.values, memb.storm_lat.values, memb.w,
               np.linspace(w_min, w_max, num_levels_w),
               colors=style.accent('orchid'), linewidths=1.2, zorder=19)

    ax.set_title(style.experiment_label(experiment), fontsize=fs_title, fontweight='bold')
    ax.set_xlabel('')  # xarray labels the axes from the variable names
    ax.tick_params(axis='both', which='major', labelsize=fs_tick)

# diff -------------------------
diff = members['incr'] - members['pi']
ax = axes[3]
c_diff = (diff.q*1000).plot.contourf(levels=lev_number, cmap=style.CMAP_DIV, ax=ax,
                                     add_colorbar=False, vmin=-1, vmax=1)

# isolines, in ink here: the centre of CMAP_DIV is white
cs = ax.contour(diff.storm_lon.values, diff.storm_lat.values, diff.z,
                np.linspace(-600, 600, num_levels_z),
                colors=style.CONTOUR_DARK, linewidths=0.8, zorder=16)
plt.clabel(cs, fmt='%d', fontsize=fs_tick)

ax.set_title('diff(pi, fut)', fontsize=fs_title, fontweight='bold')
ax.set_xlabel('')
ax.tick_params(axis='both', which='major', labelsize=fs_tick)

# axes -------------------------
axes[0].set_ylabel('lat from storm centre', fontsize=fs_label)
for ax in axes[1:]:
    ax.set_ylabel('')

# y axis number formatting
axes[0].yaxis.set_major_locator(MultipleLocator(5))
axes[0].yaxis.set_major_formatter(FormatStrFormatter('%d'))
axes[0].yaxis.set_minor_locator(MultipleLocator(5))

fig.supxlabel('lon from storm centre', x=0.46, y=-0.03, va='top', fontsize=fs_label)

# panel labels a, b, c, d
style.panel_labels(axes)

# cbars -------------------------
plt.subplots_adjust(bottom=0.1, right=0.8, top=0.9)

cax = plt.axes([0.82, 0.1, 0.008, 0.8])  # left, bottom, width, height
cbar = plt.colorbar(c, cax=cax, format='%.1f')
cbar.set_label(label='specific humidity in g/kg', size=fs_label)
cbar.ax.tick_params(labelsize=fs_tick)

cax_diff = plt.axes([0.9, 0.1, 0.008, 0.8])
cbar_diff = plt.colorbar(c_diff, cax=cax_diff, format='%.1f')
cbar_diff.set_label(label='spec. hum. difference in g/kg', size=fs_label)
cbar_diff.ax.tick_params(labelsize=fs_tick)

In [ ]:
# setup -------------------------
fig, axes = plt.subplots(1, 4, figsize=(20, 5), sharey=True)
vmin = -7
vmax = 7
levs = 11
ts = 0
ini = '2022-02-16'
t_min = 230
t_max = 360
num_levels_t = 15
member = 4
levs_array = np.arange(vmin, vmax+0.5, 0.5)

fs_title = style.scaled(style.FS_TITLE, fig)
fs_label = style.scaled(style.FS_LABEL, fig)
fs_tick = style.scaled(style.FS_TICK, fig)

pl_storms = {'pi': pi_storms, 'curr': curr_storms, 'incr': incr_storms}

# one panel per experiment -------------------------
members = {}
for ax, experiment in zip(axes, ['pi', 'curr', 'incr']):
    memb = pl_storms[experiment].sel(storm_lat=0, inidate=ini, timestep=ts,
                                     number=member).squeeze()
    members[experiment] = memb

    c = (memb.vo*10e3).plot.contourf(levels=levs, cmap=style.CMAP_SIGNED, ax=ax,
                                     center=0, vmax=vmax, add_colorbar=False)

    # potential temperature isolines
    cs = ax.contour(memb.storm_lon.values, memb.level.values,
                    pot_temp[experiment].sel(inidate=ini, storm_lat=0, timestep=ts)
                                        .mean(dim='number').squeeze(),
                    np.linspace(t_min, t_max, num_levels_t),
                    colors=style.CONTOUR_DARK, linewidths=0.8, zorder=16)
    plt.clabel(cs, fmt='%d', fontsize=fs_tick)

    ax.set_title(style.experiment_label(experiment), fontsize=fs_title, fontweight='bold')
    ax.set_xlabel('')  # xarray labels the axes from the variable names
    ax.set_ylim(150, 1000)
    ax.set_yscale('log')
    ax.tick_params(axis='both', which='major', labelsize=fs_tick)
    ax.invert_yaxis()

# diff -------------------------
diff = members['incr'] - members['pi']
ax = axes[3]
c_diff = (diff.vo*10e3).plot.contourf(ax=ax, levels=levs, cmap=style.CMAP_DIV,
                                      vmin=-2, vmax=2, center=0, add_colorbar=False)

cs = ax.contour(diff.storm_lon.values, diff.level.values,
                pot_temp['incr'].sel(inidate=ini, storm_lat=0, timestep=ts).mean(dim='number').squeeze()
                - pot_temp['pi'].sel(inidate=ini, storm_lat=0, timestep=ts).mean(dim='number').squeeze(),
                np.linspace(-5, 5, num_levels_t),
                colors=style.CONTOUR_DARK, linewidths=0.8, zorder=16)
plt.clabel(cs, fmt='%d', fontsize=fs_tick)

ax.set_title('diff(pi, fut)', fontsize=fs_title, fontweight='bold')
ax.set_xlabel('')
ax.set_ylim(150, 1000)
ax.set_yscale('log')
ax.tick_params(axis='both', which='major', labelsize=fs_tick)
ax.invert_yaxis()

# axes -------------------------
axes[0].set_ylabel('pressure level', fontsize=fs_label)
for ax in axes[1:]:
    ax.set_ylabel('')

# y axis number formatting: the axis is log-spaced, so it is labelled at the
# standard pressure levels rather than on a fixed step
axes[0].set_yticks([200, 300, 500, 700, 850, 1000])
axes[0].yaxis.set_major_formatter(FormatStrFormatter('%d'))
axes[0].minorticks_off()

fig.supxlabel('lon from storm centre', x=0.46, y=-0.03, va='top', fontsize=fs_label)

# panel labels continue the lettering of the humidity figure above
style.panel_labels(axes, letters='efgh')

# cbars -------------------------
plt.subplots_adjust(bottom=0.1, right=0.8, top=0.9)

cax = plt.axes([0.82, 0.1, 0.008, 0.8])  # left, bottom, width, height
cbar = plt.colorbar(c, cax=cax, format='%.1f')
cbar.set_label(label=r'vorticity in 10$^{-3}$/s', size=fs_label)
cbar.ax.tick_params(labelsize=fs_tick)

cax_diff = plt.axes([0.9, 0.1, 0.008, 0.8])
cbar_diff = plt.colorbar(c_diff, cax=cax_diff, format='%.1f')
cbar_diff.set_label(label=r'vorticity difference in 10$^{-3}$/s', size=fs_label)
cbar_diff.ax.tick_params(labelsize=fs_tick)